# Notebook 05 — Raw Multimodal 1D-CNN Baseline

This notebook trains a one-dimensional convolutional neural network directly on the synchronized raw multimodal WESAD windows generated in `04_prepare_raw_multimodal_windows.ipynb`.

## Objectives

1. Load the processed raw-window dataset from Notebook 04.
2. Train and evaluate a 1D CNN using Leave-One-Subject-Out (LOSO) validation.
3. Compare raw-signal deep-learning performance with the classical feature-based baselines.
4. Save evaluation metrics, figures, training histories, predictions, and summary tables.

## Expected dataset structure

```text
X shape:        (883, 1920, 8)
y shape:        (883,)
subjects shape: (883,)
channels:       ACC_x, ACC_y, ACC_z, ECG, EDA, EMG, RESP, TEMP
```

Each input sample contains a 60-second multimodal window sampled at 32 Hz.

In [ ]:
# ============================================================
# STEP 1. Imports and reproducibility
# ============================================================

import os
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import backend as K

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices('GPU'))


In [ ]:
# ============================================================
# STEP 2. Mount Google Drive and define paths
# ============================================================

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount skipped or already mounted:", e)

PROJECT_ROOT = Path('/content/drive/MyDrive/Apple_Hidden_Wellness_AI')
DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
RAW_WINDOW_DIR = PROCESSED_DIR / 'raw_multimodal_windows'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'
MODEL_DIR = PROJECT_ROOT / 'models' / 'cnn_1d_loso'

for d in [RESULTS_DIR, FIGURES_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RAW_WINDOW_NPZ = RAW_WINDOW_DIR / 'wesad_raw_multimodal_windows_32hz_60s_30s.npz'
METADATA_JSON = RAW_WINDOW_DIR / 'wesad_raw_multimodal_windows_metadata.json'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('RAW_WINDOW_NPZ:', RAW_WINDOW_NPZ)
print('Exists:', RAW_WINDOW_NPZ.exists())


In [ ]:
# ============================================================
# STEP 3. Load raw-window dataset from 04A
# ============================================================

if not RAW_WINDOW_NPZ.exists():
    raise FileNotFoundError(
        f"Could not find {RAW_WINDOW_NPZ}. Run Notebook 04A first and confirm the output path."
    )

loaded = np.load(RAW_WINDOW_NPZ, allow_pickle=True)
X = loaded['X'].astype(np.float32)
y = loaded['y'].astype(np.int64)
subjects = loaded['subjects'].astype(str)

if 'channels' in loaded.files:
    channels = loaded['channels'].astype(str).tolist()
else:
    channels = ['ACC_x', 'ACC_y', 'ACC_z', 'ECG', 'EDA', 'EMG', 'RESP', 'TEMP']

metadata = {}
if METADATA_JSON.exists():
    with open(METADATA_JSON, 'r') as f:
        metadata = json.load(f)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('subjects shape:', subjects.shape)
print('channels:', channels)
print('unique subjects:', sorted(np.unique(subjects)))
print('class counts:', dict(zip(*np.unique(y, return_counts=True))))

assert X.ndim == 3, 'X must be 3D: samples, time, channels'
assert len(X) == len(y) == len(subjects), 'X/y/subjects length mismatch'
assert set(np.unique(y)).issubset({0, 1}), 'Expected binary labels: 0 baseline, 1 stress'


In [ ]:
# ============================================================
# STEP 4. Dataset summary table and visual check
# ============================================================

def subject_sort_key(s):
    return int(str(s).replace('S', ''))

subject_order = sorted(np.unique(subjects), key=subject_sort_key)

summary_rows = []
for sid in subject_order:
    mask = subjects == sid
    summary_rows.append({
        'subject': sid,
        'n_windows': int(mask.sum()),
        'baseline_windows': int(np.sum(mask & (y == 0))),
        'stress_windows': int(np.sum(mask & (y == 1))),
    })

raw_summary_df = pd.DataFrame(summary_rows)
raw_summary_df


In [ ]:
# ============================================================
# STEP 5. Plot dataset distributions
# ============================================================

plt.figure(figsize=(10, 4))
plt.bar(raw_summary_df['subject'], raw_summary_df['n_windows'])
plt.xlabel('Subject')
plt.ylabel('Number of windows')
plt.title('Raw WESAD Windows by Subject')
plt.grid(True, axis='y')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cnn05_raw_windows_by_subject.png', dpi=300, bbox_inches='tight')
plt.show()

class_counts = pd.Series(y).value_counts().sort_index()
plt.figure(figsize=(5, 4))
plt.bar(['Baseline / Non-stress', 'Stress'], [class_counts.get(0, 0), class_counts.get(1, 0)])
plt.ylabel('Number of windows')
plt.title('Raw WESAD Class Distribution')
plt.grid(True, axis='y')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cnn05_raw_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()


## Model design

The 1D CNN serves as a raw-signal deep-learning baseline for comparison with the feature-based models and the latent-state architecture developed in the subsequent analysis.

The model evaluates whether stress-related physiological patterns can be learned directly from synchronized raw multimodal signals without manually engineered statistical features.

The architecture uses one-dimensional convolutional blocks, batch normalization, global average pooling, dropout, and class-weighted training.

In [ ]:
# ============================================================
# STEP 6. Define 1D CNN model and helper functions
# ============================================================

def build_1d_cnn(input_shape, learning_rate=1e-3):
    # Compact 1D CNN baseline for raw multimodal WESAD windows.
    inputs = keras.Input(shape=input_shape, name='raw_multimodal_window')

    x = layers.Conv1D(32, kernel_size=7, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.20)(x)

    x = layers.Conv1D(64, kernel_size=5, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv1D(128, kernel_size=3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.30)(x)

    x = layers.Conv1D(128, kernel_size=3, padding='same', activation='relu', name='last_conv')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(1, activation='sigmoid', name='stress_probability')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='WESAD_1D_CNN_Baseline')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[
            keras.metrics.BinaryAccuracy(name='accuracy'),
            keras.metrics.AUC(name='auc'),
        ],
    )
    return model


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }
    try:
        metrics['roc_auc'] = roc_auc_score(y_true, y_prob)
    except ValueError:
        metrics['roc_auc'] = np.nan
    return metrics, y_pred


def make_callbacks(fold_model_path):
    return [
        keras.callbacks.EarlyStopping(
            monitor='val_auc', mode='max', patience=8,
            restore_best_weights=True, verbose=0,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_auc', mode='max', factor=0.5,
            patience=4, min_lr=1e-5, verbose=0,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=str(fold_model_path), monitor='val_auc', mode='max',
            save_best_only=True, save_weights_only=False, verbose=0,
        ),
    ]

input_shape = X.shape[1:]
test_model = build_1d_cnn(input_shape)
test_model.summary()
K.clear_session()


In [ ]:
# ============================================================
# STEP 7. LOSO 1D CNN training
# ============================================================

MAX_EPOCHS = 40
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
THRESHOLD = 0.5

# QUICK_RUN reduces the number of LOSO folds for implementation checks.
# The full subject-independent evaluation uses QUICK_RUN=False.
QUICK_RUN = False
MAX_FOLDS_FOR_QUICK_RUN = 2

all_true = []
all_prob = []
all_pred = []
all_subject = []
fold_results = []
history_rows = []

fold_subjects = subject_order[:MAX_FOLDS_FOR_QUICK_RUN] if QUICK_RUN else subject_order

for fold_idx, test_subject in enumerate(fold_subjects, start=1):
    print("\n" + "=" * 70)
    print(f"LOSO Fold {fold_idx}/{len(fold_subjects)} | Held-out subject: {test_subject}")
    print("=" * 70)

    test_mask = subjects == test_subject
    trainval_subjects = [s for s in subject_order if s != test_subject]

    # Subject-independent validation: use the last two non-test subjects.
    val_subjects = trainval_subjects[-2:]
    train_subjects = trainval_subjects[:-2]

    train_mask = np.isin(subjects, train_subjects)
    val_mask = np.isin(subjects, val_subjects)

    X_train, y_train = X[train_mask], y[train_mask]
    X_val, y_val = X[val_mask], y[val_mask]
    X_test, y_test = X[test_mask], y[test_mask]

    print('Train subjects:', train_subjects)
    print('Val subjects  :', val_subjects)
    print('Test subject  :', test_subject)
    print('Train shape:', X_train.shape, 'Val shape:', X_val.shape, 'Test shape:', X_test.shape)
    print('Train class counts:', dict(zip(*np.unique(y_train, return_counts=True))))
    print('Val class counts  :', dict(zip(*np.unique(y_val, return_counts=True))))
    print('Test class counts :', dict(zip(*np.unique(y_test, return_counts=True))))

    classes = np.array([0, 1])
    class_weights_arr = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weight = {int(c): float(w) for c, w in zip(classes, class_weights_arr)}
    print('Class weights:', class_weight)

    K.clear_session()
    tf.random.set_seed(SEED + fold_idx)
    model = build_1d_cnn(input_shape=X.shape[1:], learning_rate=LEARNING_RATE)
    fold_model_path = MODEL_DIR / f'cnn_loso_{test_subject}.keras'

    start_time = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight,
        callbacks=make_callbacks(fold_model_path),
        verbose=0,
    )
    train_time_sec = time.time() - start_time

    pred_start = time.time()
    y_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).reshape(-1)
    inference_time_sec = time.time() - pred_start

    metrics, y_pred = compute_metrics(y_test, y_prob, threshold=THRESHOLD)

    fold_results.append({
        'fold': fold_idx,
        'test_subject': test_subject,
        'val_subjects': ','.join(val_subjects),
        'n_train': int(len(y_train)),
        'n_val': int(len(y_val)),
        'n_test': int(len(y_test)),
        'train_baseline': int(np.sum(y_train == 0)),
        'train_stress': int(np.sum(y_train == 1)),
        'test_baseline': int(np.sum(y_test == 0)),
        'test_stress': int(np.sum(y_test == 1)),
        'accuracy': metrics['accuracy'],
        'precision': metrics['precision'],
        'recall': metrics['recall'],
        'f1': metrics['f1'],
        'roc_auc': metrics['roc_auc'],
        'epochs_trained': int(len(history.history['loss'])),
        'train_time_sec': float(train_time_sec),
        'inference_time_sec': float(inference_time_sec),
        'model_path': str(fold_model_path),
    })

    print(
        f"Fold result | Accuracy={metrics['accuracy']:.3f} | "
        f"F1={metrics['f1']:.3f} | AUC={metrics['roc_auc']:.3f} | "
        f"Epochs={len(history.history['loss'])}"
    )

    all_true.extend(y_test.tolist())
    all_prob.extend(y_prob.tolist())
    all_pred.extend(y_pred.tolist())
    all_subject.extend([test_subject] * len(y_test))

    for epoch_idx in range(len(history.history['loss'])):
        row = {'fold': fold_idx, 'test_subject': test_subject, 'epoch': epoch_idx + 1}
        for key, values in history.history.items():
            row[key] = values[epoch_idx]
        history_rows.append(row)

fold_results_df = pd.DataFrame(fold_results)
pooled_predictions_df = pd.DataFrame({
    'subject': all_subject,
    'y_true': all_true,
    'y_pred': all_pred,
    'y_prob_stress': all_prob,
})
history_df = pd.DataFrame(history_rows)

fold_results_df


In [ ]:
# ============================================================
# STEP 8. Pooled LOSO evaluation
# ============================================================

all_true_arr = np.asarray(all_true).astype(int)
all_prob_arr = np.asarray(all_prob).astype(float)
all_pred_arr = np.asarray(all_pred).astype(int)

pooled_metrics, _ = compute_metrics(all_true_arr, all_prob_arr, threshold=THRESHOLD)

pooled_summary_df = pd.DataFrame([{
    'model': '1D CNN raw multimodal LOSO',
    'n_windows': int(len(all_true_arr)),
    'n_subjects': int(len(np.unique(all_subject))),
    'accuracy': pooled_metrics['accuracy'],
    'precision': pooled_metrics['precision'],
    'recall': pooled_metrics['recall'],
    'f1': pooled_metrics['f1'],
    'roc_auc': pooled_metrics['roc_auc'],
    'threshold': THRESHOLD,
    'max_epochs': MAX_EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
}])

print('Pooled LOSO performance:')
print(pooled_summary_df.T)
print('\nClassification report:')
print(classification_report(all_true_arr, all_pred_arr, target_names=['Baseline', 'Stress'], zero_division=0))


In [ ]:
# ============================================================
# STEP 9. Plot F1 by held-out subject
# ============================================================

plot_df = fold_results_df.sort_values('f1')

plt.figure(figsize=(11, 4))
plt.bar(plot_df['test_subject'], plot_df['f1'])
plt.xlabel('Held-out subject')
plt.ylabel('F1 score')
plt.ylim(0, 1.05)
plt.title('LOSO 1D CNN: F1 by Held-Out Subject')
plt.grid(True, axis='y')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cnn05_loso_f1_by_subject.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# STEP 10. Pooled ROC curve
# ============================================================

fpr, tpr, thresholds = roc_curve(all_true_arr, all_prob_arr)
auc_value = roc_auc_score(all_true_arr, all_prob_arr)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f'1D CNN LOSO, AUC = {auc_value:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--', label='Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('LOSO 1D CNN: Pooled ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cnn05_loso_pooled_roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# STEP 11. Pooled confusion matrix
# ============================================================

cm = confusion_matrix(all_true_arr, all_pred_arr, labels=[0, 1])

fig, ax = plt.subplots(figsize=(5, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Baseline', 'Stress'])
disp.plot(ax=ax, values_format='d', colorbar=True)
ax.set_title('LOSO 1D CNN: Pooled Confusion Matrix')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cnn05_loso_pooled_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

cm_df = pd.DataFrame(cm, index=['true_baseline', 'true_stress'], columns=['pred_baseline', 'pred_stress'])
cm_df


In [ ]:
# ============================================================
# STEP 12. Training curves summary
# ============================================================

if len(history_df) > 0:
    curve_cols = [c for c in ['loss', 'val_loss', 'accuracy', 'val_accuracy', 'auc', 'val_auc'] if c in history_df.columns]
    mean_history_df = history_df.groupby('epoch')[curve_cols].mean().reset_index()

    for metric_name in ['loss', 'auc']:
        val_name = f'val_{metric_name}'
        if metric_name in mean_history_df.columns and val_name in mean_history_df.columns:
            plt.figure(figsize=(7, 4))
            plt.plot(mean_history_df['epoch'], mean_history_df[metric_name], label=f'Train {metric_name}')
            plt.plot(mean_history_df['epoch'], mean_history_df[val_name], label=f'Validation {metric_name}')
            plt.xlabel('Epoch')
            plt.ylabel(metric_name.upper())
            plt.title(f'1D CNN Mean Training Curve: {metric_name.upper()}')
            plt.legend()
            plt.grid(True)
            plt.tight_layout()
            plt.savefig(FIGURES_DIR / f'cnn05_mean_training_curve_{metric_name}.png', dpi=300, bbox_inches='tight')
            plt.show()
else:
    print('No history rows were saved.')


In [ ]:
# ============================================================
# STEP 13. Save all Notebook 05 results
# ============================================================

fold_results_path = RESULTS_DIR / 'wesad_1dcnn_loso_by_subject_results.csv'
pooled_predictions_path = RESULTS_DIR / 'wesad_1dcnn_loso_pooled_predictions.csv'
pooled_summary_path = RESULTS_DIR / 'wesad_1dcnn_loso_pooled_summary.csv'
history_path = RESULTS_DIR / 'wesad_1dcnn_loso_training_history.csv'
confusion_matrix_path = RESULTS_DIR / 'wesad_1dcnn_loso_confusion_matrix.csv'
manifest_path = RESULTS_DIR / 'wesad_05_1dcnn_output_manifest.csv'

fold_results_df.to_csv(fold_results_path, index=False)
pooled_predictions_df.to_csv(pooled_predictions_path, index=False)
pooled_summary_df.to_csv(pooled_summary_path, index=False)
history_df.to_csv(history_path, index=False)
cm_df.to_csv(confusion_matrix_path)

config = {
    'notebook': '05_wesad_raw_multimodal_1dcnn_baseline',
    'raw_window_npz': str(RAW_WINDOW_NPZ),
    'metadata_json': str(METADATA_JSON),
    'n_windows': int(X.shape[0]),
    'window_samples': int(X.shape[1]),
    'n_channels': int(X.shape[2]),
    'channels': channels,
    'subjects': subject_order,
    'class_counts': {str(k): int(v) for k, v in zip(*np.unique(y, return_counts=True))},
    'evaluation': 'Leave-One-Subject-Out',
    'validation_strategy': 'two training subjects held out for validation in each fold',
    'architecture': 'compact Conv1D CNN with batch normalization, dropout, and global average pooling',
    'max_epochs': MAX_EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'threshold': THRESHOLD,
    'quick_run': QUICK_RUN,
    'pooled_metrics': {k: float(v) for k, v in pooled_metrics.items()},
}

config_path = RESULTS_DIR / 'wesad_1dcnn_loso_config_and_summary.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

saved_files = [
    fold_results_path,
    pooled_predictions_path,
    pooled_summary_path,
    history_path,
    confusion_matrix_path,
    config_path,
    FIGURES_DIR / 'cnn05_raw_windows_by_subject.png',
    FIGURES_DIR / 'cnn05_raw_class_distribution.png',
    FIGURES_DIR / 'cnn05_loso_f1_by_subject.png',
    FIGURES_DIR / 'cnn05_loso_pooled_roc_curve.png',
    FIGURES_DIR / 'cnn05_loso_pooled_confusion_matrix.png',
]

manifest_df = pd.DataFrame({'saved_file': [str(p) for p in saved_files], 'exists': [Path(p).exists() for p in saved_files]})
manifest_df.to_csv(manifest_path, index=False)

print('=' * 70)
print('Notebook 05 results saved successfully.')
print('Results directory:', RESULTS_DIR)
print('Figures directory:', FIGURES_DIR)
print('Model directory:', MODEL_DIR)
print('=' * 70)
print(manifest_df)


## Interpretation

This notebook establishes a subject-independent 1D-CNN baseline using synchronized raw multimodal physiological signals under Leave-One-Subject-Out (LOSO) evaluation.

The resulting performance provides a reference point for comparison with the classical feature-based models developed in Notebook 03. Differences in performance should be interpreted in the context of the input representation: the 1D-CNN learns directly from raw physiological time-series, whereas the classical models use manually engineered statistical features.

Notebook 06 extends this analysis by learning a shared latent physiological representation through a multi-objective architecture that jointly performs stress classification, masked-sensor reconstruction, and future-state prediction.

```text
Raw multimodal signals
        ↓
Shared encoder
        ↓
Latent physiological representation
        ↓
Hidden Wellness Index
        ↓
Stress classification
+ masked-sensor reconstruction
+ future-state prediction
```